# alpha-mipt-rag — full pipeline run

Интерактивный прогон RAG поверх корпуса `alfabank.ru` → отправляемый CSV.

**Перед запуском убедись, что:**
1. `uv sync` отработал и kernel запущен в `.venv`.
2. Артефакты индекса лежат в `artifacts/` (если нет — ячейка ниже соберёт их).
3. GGUF лежит в `models/qwen2.5-7b-instruct-q4_k_m-{00001,00002}-of-00002.gguf`.

**Полный прогон:** Kernel → Restart & Run All. ≈ 14 с/вопрос × 6977 ≈ 27 ч. Для итераций — `N_QUESTIONS = 20`.

## 0. Окружение

`OMP_NUM_THREADS=1` и `KMP_DUPLICATE_LIB_OK=TRUE` **обязаны** быть выставлены ДО импорта `torch`/`bm25s`, иначе на macOS они дерутся за libomp и падает segfault.

In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo_root:", REPO_ROOT)
print("OMP_NUM_THREADS =", os.environ["OMP_NUM_THREADS"])

repo_root: /Users/akeksandr/Projects/alpha_mipt_rag
OMP_NUM_THREADS = 1


In [2]:
import time

import polars as pl
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer

from rag.config import load_config, seed_everything
from rag.chunking import load_chunks
from rag.retrieval.dense import DenseRetriever
from rag.retrieval.bm25 import BM25Retriever
from rag.retrieval.hybrid import HybridRetriever
from rag.rerank import Reranker
from rag.grounding import assemble_context
from rag.generation import Generator
from rag.pipeline import Pipeline
from rag.submission import write_submission

print("torch:", torch.__version__, "mps:", torch.backends.mps.is_available())

/Users/akeksandr/Projects/alpha_mipt_rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.12.0 mps: True


## 1. Конфиг + сид

Источник истины — `config.yaml`. Меняешь параметры там → перезапускаешь ячейку.

In [3]:
cfg = load_config()
seed_everything(cfg.seed)

print("seed:", cfg.seed)
print("embedder:", cfg.embedder.model)
print("reranker:", cfg.reranker.model, f"(top-k={cfg.reranker.top_k})")
print("generator:", cfg.generator.model_path)
print("  temp:", cfg.generator.temperature, "max_new_tokens:", cfg.generator.max_new_tokens)
print("answer_max_chars:", cfg.length.answer_max_chars)

seed: 42
embedder: intfloat/multilingual-e5-base
reranker: BAAI/bge-reranker-v2-m3 (top-k=5)
generator: models/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf
  temp: 0.2 max_new_tokens: 384
answer_max_chars: 600


## 2. Индекс

Проверяем артефакты, при отсутствии пересобираем (`load_and_clean → chunk → FAISS + BM25`).

In [4]:
needed = [
    cfg.resolve(cfg.paths.chunks_parquet),
    cfg.resolve(cfg.paths.faiss_index),
    cfg.resolve(cfg.paths.bm25_pickle),
]
missing = [p for p in needed if not p.exists()]
if missing:
    print("missing artifacts:", [p.name for p in missing])
    print("rebuilding index — ~3-4 минуты…")
    from rag.chunking import chunk_dataframe, save_chunks
    from rag.data import load_and_clean

    docs = load_and_clean(cfg.resolve(cfg.paths.websites_csv), cfg.cleaning)
    chunks_df = chunk_dataframe(docs, cfg.chunking)
    save_chunks(chunks_df, cfg.resolve(cfg.paths.chunks_parquet))

    db = DenseRetriever(cfg.embedder)
    db.build_index(chunks_df)
    db.save(cfg.resolve(cfg.paths.faiss_index))

    bb = BM25Retriever()
    bb.build_index(chunks_df)
    bb.save(cfg.resolve(cfg.paths.bm25_pickle))
    del db, bb
else:
    print("all artifacts present:")
    for p in needed:
        print(f"  {p.name}: {p.stat().st_size / 1024 / 1024:.1f} MB")

all artifacts present:
  chunks.parquet: 3.7 MB
  faiss.index: 21.9 MB
  bm25.pkl: 8.9 MB


## 3. Загрузка модулей пайплайна

Самое долгое — Llama (GGUF в Metal, ~10-20 c) и bge-reranker (~5 c).

In [5]:
chunks = load_chunks(cfg.resolve(cfg.paths.chunks_parquet))
print(f"chunks: {chunks.height} rows, cols={chunks.columns}")

dense = DenseRetriever(cfg.embedder)
dense.load(cfg.resolve(cfg.paths.faiss_index))
print(f"faiss: dim={dense.dim}, ntotal={dense.index.ntotal}")

bm25 = BM25Retriever()
bm25.load(cfg.resolve(cfg.paths.bm25_pickle))
print("bm25: loaded")

chunks: 7475 rows, cols=['chunk_id', 'web_id', 'url', 'title', 'text', 'n_tokens']


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8511.97it/s]


faiss: dim=768, ntotal=7475
bm25: loaded


In [6]:
reranker = Reranker(cfg.reranker)
print("reranker loaded on", cfg.reranker.device)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 9664.01it/s]


reranker loaded on mps


In [ ]:
generator = Generator(cfg.generator, seed=cfg.seed)
print("generator loaded:", cfg.generator.model_path)/

llama_context: n_ctx_seq (8192) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


generator loaded: models/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf


In [8]:
tokenizer = AutoTokenizer.from_pretrained(cfg.chunking.tokenizer_model)
pipeline = Pipeline(cfg, chunks, dense, bm25, reranker, generator, tokenizer)
print("pipeline ready")

pipeline ready


## 4. Дымовой тест на одном вопросе

Заглядываем во внутренности: hybrid → rerank → context → answer.

In [9]:
DEBUG_QUERY = "Как открыть карту Альфа-Банка?"

hybrid = HybridRetriever(dense, bm25, cfg.retrieval, chunks)
fused = hybrid.search([DEBUG_QUERY])[0]
print(f"hybrid: {len(fused)} fused chunks")
print("top-3 by fused score:")
for c in fused[:3]:
    print(f"  score={c.score:.4f}  {c.url[:80]}")

reranked = reranker.rerank(DEBUG_QUERY, fused)
print(f"\nreranker: top-{len(reranked)} после rerank")
for c in reranked:
    print(f"  score={c.score:.4f}  {c.url[:80]}")

ctx = assemble_context("debug", DEBUG_QUERY, reranked, cfg.grounding, tokenizer)
print(f"\ncontext: {len(ctx.chunks)} chunks, {len(ctx.context_str)} chars")
print("--- preview ---")
print(ctx.context_str[:400])
print("...")

hybrid: 60 fused chunks
top-3 by fused score:
  score=0.0280  https://alfabank.ru/make-money/deposits/vklady-s-popolneniem/
  score=0.0273  https://alfabank.ru/everyday/debit-cards/alfacard/
  score=0.0271  https://alfabank.ru/everyday/debit-cards/youth-card/


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (523 > 512). Running this sequence through the model will result in indexing errors



reranker: top-5 после rerank
  score=0.9986  https://alfabank.ru/help/t/retail/debitcards/vipusk-perevipusk-i-blokirovka-kart
  score=0.9979  https://alfabank.ru/help/t/corp/debitcards/debetovaya-karta-aeroflot/perevipusk-
  score=0.9979  [https://alfabank.ru/help/t/retail/debitcards/debetovaya-karta-alfa-travel/perev
  score=0.9882  https://alfabank.ru/make-money/deposits/vklady-s-popolneniem/
  score=0.9872  https://alfabank.ru/corporate/rko/alfa-cash_cards/

context: 5 chunks, 8129 chars
--- preview ---
[источник 1: https://alfabank.ru/help/t/retail/debitcards/vipusk-perevipusk-i-blokirovka-karti/kak-zakazat-kartu/3153-kak-zakazat-kartu/]
Как заказать карту? Это можно сделать онлайн: — На сайте. Выберите карту, подходящую по условиям, нажмите Заказать карту и впишите свои данные. Карту откроем сразу — сможете в этот же день переводить деньги в мобильном приложении или платить в интернете. Карту д
...


In [10]:
ans = pipeline.answer("debug", DEBUG_QUERY)
print(f"answer ({len(ans.answer)} chars):")
print(ans.answer)

answer (528 chars):
Для открытия дебетовой карты Альфа-Банка можно воспользоваться следующими способами:

1. **На сайте банка**: Выберите подходящую карту, нажмите «Заказать карту» и введите свои данные. Карту откроют сразу, и вы сможете использовать её для переводов и онлайн-платежей.

2. **В мобильном приложении или Альфа-Онлайн**: Выберите «Все мои продукты → Новый счёт или продукт» и оформите карту. Также можно оформить неименную карту прямо в отделении.

3. **В отделении банка**: Покажите паспорт и оформите заявление на выпуск карты.

4.


## 5. Прогон

**Скорость ≈ 14 c/вопрос на M4 Pro.** Полные 6977 вопросов = ~27 ч.

- `N_QUESTIONS = 20` — быстрая проверка (~5 мин).
- `N_QUESTIONS = None` — полный прогон, пишет в `submission.csv`.

In [11]:
N_QUESTIONS: int | None = 20
OUTPUT_PATH = f"submissions/run_n{N_QUESTIONS}.csv" if N_QUESTIONS else cfg.paths.submission_csv

questions = pl.read_csv(cfg.resolve(cfg.paths.questions_csv), infer_schema_length=0)
if N_QUESTIONS is not None:
    questions = questions.head(N_QUESTIONS)
print(f"will run on {questions.height} questions → {OUTPUT_PATH}")

will run on 20 questions → submissions/run_n20.csv


In [12]:
answers = []
t0 = time.time()
for row in tqdm(questions.iter_rows(named=True), total=questions.height):
    answers.append(pipeline.answer(str(row["q_id"]), row["query"]))
dt = time.time() - t0
print(f"done in {dt:.1f}s ({dt / max(len(answers), 1):.2f}s/q)")

100%|██████████| 20/20 [04:19<00:00, 12.97s/it]

done in 259.3s (12.97s/q)


In [13]:
write_submission(
    answers,
    sample_path=cfg.resolve(cfg.paths.sample_submission_csv),
    out_path=cfg.resolve(OUTPUT_PATH),
)

[submission] wrote 20 rows → /Users/akeksandr/Projects/alpha_mipt_rag/submissions/run_n20.csv (cols=['q_id', 'answer_new'])


## 6. Инспекция результатов

Распределение длин + случайная выборка для ручного просмотра.

In [14]:
out_df = pl.read_csv(cfg.resolve(OUTPUT_PATH))
text_col = "answer_new" if "answer_new" in out_df.columns else "answer"
lens = out_df.with_columns(pl.col(text_col).str.len_chars().alias("len_chars"))["len_chars"]
no_data = out_df.filter(pl.col(text_col) == cfg.length.no_data_phrase).height

print(f"answers: {out_df.height}")
print(f"chars: min={lens.min()}, mean={lens.mean():.0f}, p50={lens.quantile(0.5):.0f}, p95={lens.quantile(0.95):.0f}, max={lens.max()}")
print(f"no-data: {no_data} / {out_df.height} ({100*no_data/out_df.height:.1f}%)")

answers: 20
chars: min=40, mean=142, p50=40, p95=558, max=563
no-data: 13 / 20 (65.0%)


In [16]:
SAMPLE_N = 10
sample = (
    questions.head(out_df.height)
    .with_columns(pl.Series(text_col, out_df[text_col]))
    .sample(n=min(SAMPLE_N, out_df.height), seed=cfg.seed)
)
for r in sample.iter_rows(named=True):
    print(f"q_id={r['q_id']}  ({len(r[text_col])} chars)")
    print(f"Q: {r['query']}")
    print(f"A: {r[text_col]}")
    print("---")

q_id=3  (40 chars)
Q: Мне не приходят коды для подтверждения данной операции
A: В контексте нет данных по этому вопросу.
---
q_id=4  (40 chars)
Q: Оформила рассрочку ,но уведомлений никаких не пришло
A: В контексте нет данных по этому вопросу.
---
q_id=9  (40 chars)
Q: Подскажите, где я могу увидеть дату начала пользование кредитной карты?
A: В контексте нет данных по этому вопросу.
---
q_id=10  (40 chars)
Q: Не приходит смс
A: В контексте нет данных по этому вопросу.
---
q_id=11  (40 chars)
Q: Не могу видеть на экране
A: В контексте нет данных по этому вопросу.
---
q_id=12  (40 chars)
Q: Не могу войти в альфа онлайн по номеру телефона
A: В контексте нет данных по этому вопросу.
---
q_id=13  (40 chars)
Q: Я заказала кредитную карту и мне одобрили почему мне не приходят СМС чтобы получить карту в банке
A: В контексте нет данных по этому вопросу.
---
q_id=16  (40 chars)
Q: Можно узнать, я вчера пополнила сумму на 0000 за кредит? Куда списался платеж ? За кредит или налог ?Что теперь нужн

## 7. Pre-submission checklist (из CLAUDE.md)

**Не отправляй CSV, не пройдясь по этому списку.** На день — 3 загрузки.

- [ ] Локальный Recall-L посчитан (или сделан ручной spot-check на образце).
- [ ] Полные `questions.csv` прогнаны без падений и пустых ответов.
- [ ] Распределение `l_a/l_r` проверено; хвостов ≥3× нет.
- [ ] no-data ответы выглядят разумно (не для очевидных вопросов).
- [ ] CSV совпадает с `sample_submission.csv` по колонкам/UTF-8/разделителю.
- [ ] Сиды, версии моделей и `config.yaml` залогированы.
- [ ] Никаких сетевых вызовов на пути инференса (веса локальные).